In [2]:
import kagglehub

path = kagglehub.dataset_download("adithyachalla/waste-classification")

print("Path to dataset files:", path)

/Users/jimmyzhou/Desktop/waste-cnn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 657M/657M [00:36<00:00, 18.9MB/s] 

Extracting files...


Path to dataset files: /Users/jimmyzhou/.cache/kagglehub/datasets/adithyachalla/waste-classification/versions/1


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

In [4]:

import random
from pathlib import Path
import shutil

src_root = Path("datasets/waste") 
out_root = Path("datasets/waste_split") 
splits = {"train": 0.7, "val": 0.15, "test": 0.15}
seed = 42
use_symlink = False  
allowed_exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}


def split_dataset(src_root: Path, out_root: Path, splits: dict, seed: int = 42, copy: bool = True):
    random.seed(seed)
    out_root.mkdir(parents=True, exist_ok=True)

    total_frac = sum(splits.values())
    if not (0.999 <= total_frac <= 1.001):
        raise ValueError(f"splits must sum to 1.0")

    classes = [p for p in sorted(src_root.iterdir()) if p.is_dir()]
    if not classes:
        raise FileNotFoundError(f"No class subdirectories found in {src_root}")

    summary = {"train":0,"val":0,"test":0}

    for cls_dir in classes:
        images = [p for p in sorted(cls_dir.iterdir()) if p.suffix.lower() in allowed_exts]
        if not images:
            print(f"Warning: no images found for class {cls_dir.name}")
            continue

        random.shuffle(images)
        n = len(images)
        n_train = int(n * splits["train"])
        n_val = int(n * splits["val"])
        n_test = n - n_train - n_val

        sets = {
            "train": images[:n_train],
            "val": images[n_train:n_train + n_val],
            "test": images[n_train + n_val:]
        }

        for split_name, files in sets.items():
            target_dir = out_root / split_name / cls_dir.name
            target_dir.mkdir(parents=True, exist_ok=True)
            for src_file in files:
                dst_file = target_dir / src_file.name
                if copy:
                    shutil.copy2(src_file, dst_file)
                else:
                    if dst_file.exists():
                        dst_file.unlink()
                    dst_file.symlink_to(src_file.resolve())
            summary[split_name] += len(files)

    for k, v in summary.items():
        print(f"  {k}: {v} images")


if __name__ == "__main__":
    print(f"Source: {src_root}")
    print(f"Output: {out_root}")
    split_dataset(src_root, out_root, splits, seed=seed, copy=not use_symlink)


Source: datasets/waste
Output: datasets/waste_split
  train: 3323 images
  val: 710 images
  test: 719 images
  train: 3323 images
  val: 710 images
  test: 719 images
